In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from scipy.spatial import Delaunay


file_name = 'knotinfo (1).csv'
df = pd.read_csv(file_name)

bridge2_df = df[df['Bridge Index'] == 2].copy()
bridge2_df['Alexander Degree'] = bridge2_df['Genus-3D'] * 2

x_data = bridge2_df['Crossing Number']
y_data = bridge2_df['Alexander Degree']
z_data = bridge2_df['Braid Index']
labels = bridge2_df['Name']

fig = go.Figure()


fig.add_trace(go.Scatter3d(
    x=x_data, y=y_data, z=z_data,
    mode="markers",
    marker=dict(
        size=4,
        color="gray",
        opacity=0.8
    ),
    text=labels,
    hoverinfo="text+x+y+z",
    name="2-Bridge Knots"
))


unique_degrees = sorted(bridge2_df['Alexander Degree'].unique())

palette = ['rgb(31, 119, 180)', 'rgb(255, 127, 14)', 'rgb(44, 160, 44)',
           'rgb(214, 39, 40)', 'rgb(148, 103, 189)', 'rgb(140, 86, 75)']

for i, deg in enumerate(unique_degrees):
    subset = bridge2_df[bridge2_df['Alexander Degree'] == deg]
    if len(subset) < 3: continue

    curr_x, curr_z = subset['Crossing Number'].values, subset['Braid Index'].values
    try:

        tri = Delaunay(np.column_stack((curr_x, curr_z)))
        fig.add_trace(go.Mesh3d(
            x=curr_x, y=[deg] * len(subset), z=curr_z,
            i=tri.simplices[:, 0], j=tri.simplices[:, 1], k=tri.simplices[:, 2],
            color=palette[i % len(palette)],
            opacity=0.6,
            name=f"Degree {deg} Surface",
            showlegend=True,
            hoverinfo="skip"
        ))
    except: continue


x_len, y_len, z_len = x_data.max() * 1.15, y_data.max() * 1.15, z_data.max() * 1.15
axis_style = dict(color="#333333", width=7)


fig.add_trace(go.Scatter3d(x=[0, x_len], y=[0, 0], z=[0, 0], mode="lines", line=axis_style, showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter3d(x=[0, 0], y=[0, y_len], z=[0, 0], mode="lines", line=axis_style, showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter3d(x=[0, 0], y=[0, 0], z=[0, z_len], mode="lines", line=axis_style, showlegend=False, hoverinfo="skip"))

def add_arrow(fig, x, y, z, u, v, w):
    fig.add_trace(go.Cone(
        x=[x], y=[y], z=[z], u=[u], v=[v], w=[w],
        anchor="tip", showscale=False,
        colorscale=[[0, "#333333"], [1, "#333333"]],
        sizemode="absolute",
        sizeref=max(x_len, y_len, z_len) * 0.1,
        hoverinfo="skip"
    ))


add_arrow(fig, x_len, 0, 0, 1, 0, 0)
add_arrow(fig, 0, y_len, 0, 0, 1, 0)
add_arrow(fig, 0, 0, z_len, 0, 0, 1)

fig.update_layout(
    scene=dict(
        xaxis=dict(title="Crossing Number (X)", showbackground=False, showgrid=False, range=[0, x_len*1.05]),
        yaxis=dict(title="Alexander Degree (Y)", showbackground=False, showgrid=False, range=[0, y_len*1.05]),
        zaxis=dict(title="Braid Index (Z)", showbackground=False, showgrid=False, range=[0, z_len*1.05]),
        bgcolor="white",
        aspectmode="cube"
    ),
    paper_bgcolor="white",
    margin=dict(l=0, r=0, t=30, b=0),
    legend=dict(
        title="<b>Layers & Data</b>",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="Gray",
        borderwidth=1
    )
)

pio.write_html(fig, "bridge_2_knots_gray_dots.html")
fig.show()

